# Ejercicios UD1 · Bloque 2 — Análisis exploratorio de datos (AED)

**Módulo:** Depuración, Limpieza y Clasificación de Datos (5109)
**Unidad Didáctica:** UD1 — Análisis exploratorio de datos: estructura, variables y calidad
**Alumno/a:**
**Fecha de entrega:**

---

Este cuaderno recoge los ejercicios del **Bloque 2** de la UD1. Están organizados por sesiones, igual que los cuadernos de teoría:

| Sesión | Cuaderno de teoría | Ejercicios |
|---|---|---|
| 1 | `B2_S1_Concepto_e_Inspeccion_Inicial` | 1.1 – 1.4 |
| 2 | `B2_S2_Variables_y_Relaciones` | 2.1 – 2.5 |
| 3 | `B2_S3_Visualizacion_y_Perfilado` | 3.1 – 3.4 |

**Todos los ejercicios trabajan sobre el mismo conjunto de datos**: pacientes estudiados por sospecha de **enfermedad cardiaca**. Así, al terminar el cuaderno habrás hecho un AED completo de principio a fin, que resumirás en una ficha de síntesis (ejercicio 3.4).

Los ejercicios se encadenan: el DataFrame `heart` que preparas en la Sesión 1 se usa en el resto del cuaderno. Algunos ejercicios son preguntas de reflexión (responde en la celda de texto) y otros son de código (escribe tu solución donde pone `TODO`).

> **Antes de empezar:** copia este cuaderno a tu carpeta `mis_cuadernos/` y trabaja sobre la copia, así `git pull` no te dará conflictos.
>
> **Cómo entregar:** reinicia el kernel y ejecuta todas las celdas (**Restart** → **Run All**), comprueba que no hay errores, guarda y sube el fichero `.ipynb` al aula virtual. No hace falta que subas los informes HTML ni las imágenes que se generan.

## Criterios de evaluación

Esta relación de ejercicios evalúa los siguientes criterios de evaluación del módulo (Real Decreto 183/2026, redacción literal):

> **RA1.** Identifica la estructura y variables en los datos en el dominio de aplicación, así como su cobertura y calidad del conjunto, aplicando técnicas exploratorias mediante análisis exploratorios de datos (AED) para la selección de atributos clave en el entrenamiento posterior.

| CE | Criterio de evaluación | Ejercicios |
|:--:|------------------------|:----------:|
| **1.b** | Se han detectado datos incorrectamente introducidos, ausencia de valores, una mala codificación de las variables o valores atípicos. | 1.2, 1.3, 1.4, 2.1, 2.4, 3.3 |
| **1.c** | Se han identificado la estructura, variables y relaciones de los conjuntos de datos aplicando técnicas exploratorias. | Todos |
| **1.d** | Se han identificado entidades, volumetrías, relaciones y atributos, para documentar cada conjunto de datos, describiendo sus características. | 1.1, 1.2, 1.4, 3.4 |

In [ ]:
# Librerías que se usan en todo el cuaderno
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

print('✅ Librerías cargadas')

## El conjunto de datos: Heart Failure Prediction

Las enfermedades cardiovasculares son la primera causa de muerte en el mundo. Detectarlas pronto, a partir de pruebas sencillas, es uno de los usos clásicos del aprendizaje automático en medicina.

El dataset **[Heart Failure Prediction](https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction)** (Kaggle, fedesoriano, 2021) se construyó **juntando cinco conjuntos de datos** del repositorio UCI, recogidos en hospitales distintos: Cleveland, Hungría, Suiza, Long Beach (VA) y Statlog. Entre todos sumaban 1.190 registros; el autor eliminó los duplicados y dejó **918 pacientes**.

Cada fila es un paciente al que se estudió por sospecha de enfermedad cardiaca. Además de datos básicos (edad, sexo, presión, colesterol, glucosa…) se le hizo un **electrocardiograma en reposo** y una **prueba de esfuerzo** (caminar o correr en una cinta mientras se registra el corazón).

| Variable | Descripción | Valores / unidades |
|---|---|---|
| `Age` | Edad del paciente | años |
| `Sex` | Sexo | `M` hombre · `F` mujer |
| `ChestPainType` | Tipo de dolor torácico | `TA` angina típica · `ATA` angina atípica · `NAP` dolor no anginoso · `ASY` asintomático |
| `RestingBP` | Presión arterial en reposo | mm Hg |
| `Cholesterol` | Colesterol en sangre | mg/dL |
| `FastingBS` | Glucosa en ayunas alta | `1` si > 120 mg/dL · `0` en otro caso |
| `RestingECG` | Electrocardiograma en reposo | `Normal` · `ST` alteración de la onda ST-T · `LVH` hipertrofia ventricular izquierda |
| `MaxHR` | Frecuencia cardiaca máxima alcanzada en la prueba de esfuerzo | pulsaciones por minuto |
| `ExerciseAngina` | Dolor de pecho (angina) provocado por el esfuerzo | `Y` sí · `N` no |
| `Oldpeak` | Desplazamiento del segmento ST durante el esfuerzo respecto al reposo | mm |
| `ST_Slope` | Pendiente del segmento ST en el pico del esfuerzo | `Up` ascendente · `Flat` plana · `Down` descendente |
| `HeartDisease` | **Variable objetivo**: diagnóstico de enfermedad cardiaca | `1` enfermo · `0` sano |

La celda siguiente descarga el fichero (solo la primera vez) a la carpeta `datos/` y lo carga. Si la descarga falla, descarga `heart.csv` desde Kaggle y guárdalo tú en `datos/heart.csv`.

> `heart_raw` es la copia **original** y no se toca en todo el cuaderno. Trabaja siempre sobre `heart`.

In [ ]:
URL_HEART = 'https://raw.githubusercontent.com/imteekay/heart-failure-prediction/main/heart.csv'
ruta_heart = Path('datos') / 'heart.csv'

if not ruta_heart.exists():
    print('Descargando heart.csv...')
    ruta_heart.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve(URL_HEART, ruta_heart)

heart_raw = pd.read_csv(ruta_heart)   # copia original: no la modifiques
heart = heart_raw.copy()              # copia de trabajo
print('✅ Dataset cargado')

---
# Sesión 1 — Concepto del AED e inspección inicial

## Ejercicio 1.1 — Primer contacto con los datos

**Parte A — Vistazo.**
Muestra, cada cosa en su celda o con `display()`:

1. Las dimensiones del dataset y los nombres de las columnas.
2. Las 5 primeras y las 5 últimas filas.
3. 8 filas elegidas al azar, con `random_state=42` para que el resultado sea reproducible.
4. El resumen de `info()`.
5. El número de filas duplicadas.

In [ ]:
# TODO: dimensiones y columnas

# TODO: primeras y últimas filas

# TODO: 8 filas al azar (random_state=42)

# TODO: info()

# TODO: filas duplicadas

**Preguntas:**

1. ¿Qué **entidad** representa cada fila? ¿Hay alguna columna que identifique al paciente?
2. El dataset junta cinco hospitales. Sin un identificador, ¿podrías saber si el mismo paciente aparece dos veces? ¿Qué te dice el resultado de `duplicated()` y qué **no** te dice?
3. Según `info()`, ¿cuántos valores ausentes hay? ¿Te fías de esa respuesta? (Lo comprobarás en el ejercicio 1.3.)
4. Las filas están ordenadas por hospital de origen, uno detrás de otro. ¿Por qué es arriesgado sacar conclusiones mirando solo `head()`?

*Escribe aquí tus respuestas:*

## Ejercicio 1.2 — Ficha de variables y tipos correctos

**Parte A — Diagnóstico.**
Construye un DataFrame `ficha` con **una fila por columna** del dataset y estas columnas:

| variable | dtype | n_unicos | ejemplos | tipo_estadistico |
|---|---|---|---|---|

- `ejemplos`: los 4 primeros valores distintos de la columna.
- `tipo_estadistico`: escríbelo tú con un diccionario. Usa una de estas etiquetas: *numérica continua*, *numérica discreta*, *categórica nominal*, *categórica ordinal* o *binaria*.

In [ ]:
# TODO: ficha de variables
# tipo_estadistico = {
#     'Age': '...',
#     ...
# }
# ficha = pd.DataFrame({...})
# ficha

**Parte B — Corrección de tipos.**

1. Calcula la memoria que ocupa `heart` (`heart.memory_usage(deep=True).sum()`).
2. Convierte a `category` las columnas de texto: `Sex`, `ChestPainType`, `RestingECG`, `ExerciseAngina`.
3. Decide si `ST_Slope` es ordinal. Si lo es, conviértela con un `pd.CategoricalDtype(..., ordered=True)` en el orden que consideres correcto.
4. Decide qué hacer con `FastingBS` y `HeartDisease`: ¿las dejas como enteros o las conviertes? Justifícalo en las preguntas.
5. Vuelve a calcular la memoria y muestra `heart.dtypes`.

In [ ]:
# TODO: memoria antes

# TODO: conversión de tipos

# TODO: memoria después y dtypes

**Preguntas:**

1. pandas ha cargado `FastingBS` como `int64`. ¿Es una variable numérica? ¿Qué significa en este caso calcular su **media**?
2. ¿Es ordinal `ST_Slope`? ¿Y `ChestPainType`? Justifica las dos respuestas.
3. ¿Cuánta memoria has ahorrado? ¿Por qué ocupa menos una columna `category` que una de texto?
4. ¿Qué decisión has tomado con `FastingBS` y `HeartDisease`? ¿Por qué?

*Escribe aquí tus respuestas:*

## Ejercicio 1.3 — Estadísticos descriptivos: ¿dónde está el error?

**Parte A — Resumen estadístico.**
Muestra `describe()` de las variables numéricas (traspuesto y redondeado a 2 decimales) y `describe()` de las variables categóricas.

In [ ]:
# TODO: describe() numéricas

# TODO: describe() categóricas

**Parte B — Valores imposibles.**

1. Cuenta cuántos **ceros** hay en cada variable numérica.
2. Cuenta cuántos valores **negativos** hay en `Oldpeak`.
3. Muestra la fila o filas con `RestingBP == 0`.
4. Dibuja un diagrama de dispersión de `Cholesterol` frente al **número de fila** (`heart.index`). Pon título y etiquetas a los ejes.

In [ ]:
# TODO: ceros por variable

# TODO: negativos en Oldpeak

# TODO: filas con RestingBP == 0

# TODO: Cholesterol frente al número de fila

**Preguntas:**

1. ¿Es posible tener un colesterol de 0 mg/dL o una presión arterial de 0 mm Hg en un paciente vivo? ¿Qué son en realidad esos ceros? ¿Por qué `info()` no los detectó?
2. ¿Qué observas en el gráfico de `Cholesterol` frente al número de fila? ¿Qué te sugiere sobre el **origen** de esos ceros?
3. `Oldpeak` tiene valores negativos. Busca qué mide esta variable y si un valor negativo tiene sentido clínico (pista: *depresión* frente a *elevación* del segmento ST). ¿Lo tratarías como un error?
4. `FastingBS` tiene muchísimos ceros. ¿Son también sospechosos? ¿Por qué?

*Escribe aquí tus respuestas:*

## Ejercicio 1.4 — Variable objetivo e informe de inspección

**Parte A — Variable objetivo.**
Muestra las frecuencias absolutas y relativas (en %) de `HeartDisease` en una sola tabla.

In [ ]:
# TODO: frecuencias de HeartDisease

**Parte B — Informe de inspección automatizado.**
Escribe una función `informe_inspeccion(df)` que devuelva un DataFrame con **una fila por columna** y estas columnas:

| dtype | nulos | pct_nulos | ceros | pct_ceros | n_unicos | pct_unicos |
|---|---|---|---|---|---|---|

Los ceros solo tienen sentido en las columnas numéricas: en las demás pon 0. Ordena el resultado de mayor a menor `pct_nulos` y, después, `pct_ceros`.

Aplícala a `heart`.

In [ ]:
# TODO: función informe_inspeccion
# def informe_inspeccion(df):
#     ...
#     return informe

# informe_inspeccion(heart)

**Parte C — Los ausentes que no se veían.**
Crea una copia `heart_diag` en la que los ceros de `Cholesterol` y `RestingBP` se sustituyan por `NaN` (solo en esas dos columnas). Vuelve a aplicar `informe_inspeccion` y dibuja un mapa de calor de los ausentes de `heart_diag`, como en el Ejemplo 8 de la teoría.

> No borres ni rellenes nada: `heart_diag` solo sirve para **diagnosticar**. El tratamiento de ausentes es del Bloque 3.

In [ ]:
# TODO: heart_diag con los ceros como NaN

# TODO: informe_inspeccion(heart_diag)

# TODO: mapa de calor de ausentes

**Preguntas:**

1. ¿Está equilibrada la variable objetivo? Un «modelo» que dijera siempre *enfermo*, ¿qué porcentaje de aciertos tendría?
2. ¿Qué porcentaje de pacientes tiene el colesterol **ausente** en realidad?
3. En el mapa de calor, ¿los ausentes aparecen repartidos al azar o agrupados? ¿Encaja con lo que viste en el ejercicio 1.3?

*Escribe aquí tus respuestas:*

---
# Sesión 2 — Análisis de variables y relaciones

## Ejercicio 2.1 — Variables numéricas: distribución y valores atípicos

Trabaja con las cinco variables numéricas continuas: `Age`, `RestingBP`, `Cholesterol`, `MaxHR` y `Oldpeak`.

**Parte A — Estadísticos.**
Construye una tabla con una fila por variable y estas columnas: media, mediana, desviación típica, mínimo, Q1, Q3, máximo, IQR y asimetría (`skew()`). Añade una fila más, `Cholesterol (sin ceros)`, calculada solo con los pacientes con `Cholesterol > 0`.

In [ ]:
NUMERICAS = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']

# TODO: tabla de estadísticos (incluida la fila «Cholesterol (sin ceros)»)

**Parte B — Histogramas y boxplots.**
En una sola figura de **2 filas × 5 columnas**, dibuja en la fila de arriba el histograma de cada variable (con la curva KDE) y debajo su boxplot horizontal.

In [ ]:
# TODO: figura 2×5 con histogramas y boxplots
# fig, axes = plt.subplots(2, 5, figsize=(20, 7))

**Parte C — Atípicos por el criterio IQR.**
Escribe una función `atipicos_iqr(serie)` que devuelva un diccionario con el límite inferior, el límite superior, el número de atípicos y su porcentaje (criterio: fuera de `[Q1 − 1,5·IQR, Q3 + 1,5·IQR]`). Aplícala a las cinco variables y a `Cholesterol` sin ceros, y presenta el resultado como una tabla.

In [ ]:
# TODO: función atipicos_iqr y tabla de resultados

**Preguntas:**

1. ¿Qué variables tienen una distribución asimétrica? ¿Hacia qué lado? Compruébalo comparando media y mediana.
2. ¿Cómo cambian la media, la desviación típica y el número de atípicos de `Cholesterol` al quitar los ceros? ¿Qué cálculo de los dos es el correcto?
3. Mira los atípicos de cada variable. ¿Cuáles crees que son **errores** y cuáles son **valores extremos reales** que hay que conservar? Pon al menos un ejemplo de cada tipo.

*Escribe aquí tus respuestas:*

## Ejercicio 2.2 — Variables categóricas: frecuencias

Trabaja con las seis variables categóricas o binarias: `Sex`, `ChestPainType`, `FastingBS`, `RestingECG`, `ExerciseAngina` y `ST_Slope`.

**Parte A — Tablas de frecuencias.**
Para cada variable, muestra una tabla con la frecuencia absoluta (`n`) y la relativa (`%`, con un decimal).

In [ ]:
CATEGORICAS = ['Sex', 'ChestPainType', 'FastingBS', 'RestingECG', 'ExerciseAngina', 'ST_Slope']

# TODO: tabla de frecuencias de cada variable

**Parte B — Gráficos de barras.**
En una figura de **2 × 3**, dibuja un gráfico de barras por variable. Las variables nominales, ordenadas de mayor a menor frecuencia; las ordinales, en su orden natural. Escribe el porcentaje encima de cada barra (pista: `ax.bar_label()`).

In [ ]:
# TODO: figura 2×3 con gráficos de barras

**Preguntas:**

1. ¿Qué categorías tienen menos del 10 % de los casos? ¿Qué problema pueden dar al entrenar un modelo?
2. ¿Qué porcentaje de los pacientes son mujeres? ¿Es representativo de la población? ¿Qué consecuencias puede tener para un modelo que se use después con mujeres?
3. La categoría más frecuente de `ChestPainType` es `ASY` (asintomático). ¿Te sorprende en un dataset de pacientes estudiados por sospecha de enfermedad cardiaca? Plantea una explicación.

*Escribe aquí tus respuestas:*

## Ejercicio 2.3 — Variables categóricas frente a la variable objetivo

**Parte A — Tasa de enfermedad por categoría.**
Para cada variable de `CATEGORICAS`, calcula el número de pacientes y el **porcentaje de enfermos** (`HeartDisease = 1`) en cada categoría. Muestra también la tabla de contingencia de `ChestPainType` × `HeartDisease` con los totales (`margins=True`), primero en valores absolutos y después en porcentaje por fila.

In [ ]:
# TODO: tasa de enfermedad por categoría

# TODO: crosstab ChestPainType × HeartDisease (absoluta y en % por fila)

**Parte B — Gráfico.**
En una figura de **2 × 3**, dibuja para cada variable un gráfico de barras con el **porcentaje de enfermos** por categoría. Añade en todos los gráficos una línea horizontal discontinua con el porcentaje de enfermos de todo el dataset.

In [ ]:
# TODO: figura 2×3 con la tasa de enfermedad por categoría

**Parte C — Ranking.**
Para cada variable, calcula la diferencia (en puntos porcentuales) entre la categoría con **más** enfermos y la que tiene **menos**. Ordena las variables de mayor a menor diferencia.

In [ ]:
# TODO: ranking de variables categóricas

**Preguntas:**

1. ¿Qué tres variables categóricas parecen más relacionadas con la enfermedad? ¿Y cuál menos?
2. Los pacientes **asintomáticos** (`ASY`) son los que más enferman. ¿Cómo lo explicas? (Recuerda quiénes son estos pacientes y por qué se les hizo la prueba.)
3. La categoría `TA` tiene solo 46 pacientes. ¿Te fías igual de su porcentaje de enfermos que del de `ASY`, con 496? ¿Por qué?

*Escribe aquí tus respuestas:*

## Ejercicio 2.4 — Correlaciones y la paradoja del colesterol

**Parte A — Matriz de correlación.**
Calcula la matriz de correlación de Pearson de las columnas numéricas (incluidas `FastingBS` y `HeartDisease`) y dibújala como mapa de calor: con los valores escritos, paleta divergente centrada en 0 y **solo el triángulo inferior** (pista: `np.triu` para crear la máscara).

In [ ]:
# TODO: matriz de correlación y heatmap

**Parte B — Correlación con la variable objetivo.**
Muestra la correlación de cada variable con `HeartDisease`, ordenada de mayor a menor **valor absoluto**.

In [ ]:
# TODO: correlaciones con HeartDisease ordenadas por valor absoluto

**Parte C — La paradoja del colesterol.**

1. Calcula la correlación entre `Cholesterol` y `HeartDisease` con **todos** los pacientes y solo con los que tienen `Cholesterol > 0`.
2. Calcula el porcentaje de enfermos entre los pacientes con `Cholesterol == 0` y entre los que tienen `Cholesterol > 0`.

In [ ]:
# TODO: correlación con y sin ceros

# TODO: porcentaje de enfermos según Cholesterol == 0

**Preguntas:**

1. Interpreta el signo de la correlación de `MaxHR` y de `Oldpeak` con la enfermedad.
2. Con todos los datos, parece que **a más colesterol, menos enfermedad**. ¿Es verdad? Explica qué está pasando con los resultados de la Parte C.
3. Si entrenaras un modelo con los datos tal como están, ¿qué «aprendería» de `Cholesterol`? ¿Funcionaría bien en un hospital que siempre mide el colesterol?
4. `FastingBS` y `HeartDisease` solo valen 0 o 1. ¿Tiene sentido incluirlas en una correlación de Pearson? ¿Qué mide en ese caso?

*Escribe aquí tus respuestas:*

## Ejercicio 2.5 — Relaciones a la vista: dispersión, violines y pairplot

**Parte A — Edad y frecuencia cardiaca máxima.**
Dibuja `MaxHR` frente a `Age`, coloreando por `HeartDisease`, con una recta de regresión para cada grupo (pista: `sns.lmplot`).

In [ ]:
# TODO: MaxHR frente a Age con recta por grupo

**Parte B — Violines.**
En una figura de 1 × 2, dibuja un violinplot de `MaxHR` y otro de `Oldpeak` según `HeartDisease`, separando por sexo dentro de cada violín (`hue='Sex', split=True`).

In [ ]:
# TODO: violinplots de MaxHR y Oldpeak

**Parte C — Pairplot.**
Dibuja un `pairplot` de `Age`, `RestingBP`, `Cholesterol`, `MaxHR` y `Oldpeak` coloreado por `HeartDisease`, mostrando solo el triángulo inferior (`corner=True`). Usa **solo los pacientes con `Cholesterol > 0` y `RestingBP > 0`**.

In [ ]:
# TODO: pairplot sin los ceros

**Preguntas:**

1. ¿Qué le pasa a la frecuencia cardiaca máxima al aumentar la edad? ¿Ocurre igual en sanos y en enfermos?
2. ¿Ves diferencias entre hombres y mujeres en los violines? ¿Cuál de las dos variables separa mejor a sanos y enfermos?
3. En el pairplot, ¿qué par de variables separa mejor las dos clases? ¿Hay alguna variable que no parezca aportar casi nada?
4. ¿Por qué hemos quitado los ceros para el pairplot? ¿Cómo se vería si no lo hubiéramos hecho?

*Escribe aquí tus respuestas:*

---
# Sesión 3 — Visualización y perfilado automático

## Ejercicio 3.1 — Dashboard del riesgo cardiaco

Crea con `gridspec` una sola figura (de unos 16 × 9 pulgadas) con **cinco paneles** distribuidos así:

```
┌───────────────────────────────┬──────────────┐
│ 1. Distribución de la edad    │ 2. % enfermos│
│    de sanos y enfermos        │ por tipo de  │
│                               │ dolor        │
├───────────────┬───────────────┼──────────────┤
│ 3. MaxHR      │ 4. % enfermos │ 5. Angina de │
│ sanos vs.     │ por pendiente │ esfuerzo por │
│ enfermos      │ ST            │ diagnóstico  │
└───────────────┴───────────────┴──────────────┘
```

Requisitos:

- Un título general (`fig.suptitle`) y un título en cada panel.
- Ejes con etiquetas en castellano y unidades cuando las haya.
- **Los mismos dos colores** para sanos y enfermos en todos los paneles. Defínelos una vez en un diccionario `COLORES` y úsalo siempre.
- En las leyendas, «Sano» y «Enfermo» en lugar de 0 y 1.
- Las categorías ordinales en su orden.
- Guarda la figura en `dashboard_corazon.png` (`dpi=150`, `bbox_inches='tight'`).

In [ ]:
# TODO: dashboard con gridspec
# COLORES = {0: '...', 1: '...'}
# fig = plt.figure(figsize=(16, 9))
# gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)
# ...

**Preguntas:**

1. Si solo pudieras enseñar este dashboard durante 30 segundos, ¿cuál sería el mensaje principal?
2. ¿Qué cambiarías si el dashboard fuera para un equipo médico en lugar de para tus compañeros de datos?

*Escribe aquí tus respuestas:*

## Ejercicio 3.2 — Comparar grupos con `catplot`, `displot` y `FacetGrid`

**Parte A.** Con `sns.catplot(kind='bar')`, dibuja la proporción de enfermos por `ChestPainType`, con un panel para cada sexo (`col='Sex'`). Ordena las categorías igual en los dos paneles.

In [ ]:
# TODO: catplot de la proporción de enfermos por tipo de dolor y sexo

**Parte B.** Con `sns.displot(kind='kde')`, dibuja la distribución de `MaxHR` de sanos y enfermos, con un panel según tengan o no angina de esfuerzo (`col='ExerciseAngina'`). Usa `common_norm=False`.

In [ ]:
# TODO: displot de MaxHR por ExerciseAngina

**Parte C.** Con `sns.FacetGrid`, crea una cuadrícula con una fila por sexo y una columna por `ST_Slope` (en su orden). En cada panel, dibuja `Oldpeak` frente a `Age` coloreando por `HeartDisease`, y añade la leyenda.

In [ ]:
# TODO: FacetGrid Sex × ST_Slope

**Preguntas:**

1. ¿Se mantiene en los dos sexos el efecto del tipo de dolor torácico? ¿Qué representan las barras negras y por qué son más largas en el panel de las mujeres?
2. ¿Qué hace `common_norm=False`? ¿Cómo se vería el gráfico de la Parte B sin ese parámetro?
3. ¿Hay algún panel del `FacetGrid` con tan pocos pacientes que no permita sacar conclusiones? ¿Cuál?

*Escribe aquí tus respuestas:*

## Ejercicio 3.3 — Perfilado automático con ydata-profiling

**Parte A — Informe completo.**
Genera un informe de perfilado de `heart_raw` con `explorative=True`, guárdalo en `informe_heart.html` y ábrelo en el navegador. Después, imprime la lista de alertas que ha detectado.

> Usamos `heart_raw` (la copia original) para ver qué detecta el informe **sin** ninguna ayuda nuestra.

In [ ]:
from ydata_profiling import ProfileReport

# TODO: informe completo de heart_raw y lista de alertas

**Parte B — Informe comparativo.**
Genera dos informes en modo `minimal=True`: uno de `heart_raw` y otro de una copia en la que los ceros de `Cholesterol` y `RestingBP` sean `NaN` (puedes reutilizar la idea de `heart_diag`, partiendo de `heart_raw`). Compáralos con `.compare()` y guarda el resultado en `informe_heart_comparativo.html`.

In [ ]:
# TODO: informes minimal y comparación

**Parte C — ¿Qué ha visto el informe y qué has visto tú?**
Completa la tabla con todos los problemas de calidad que has encontrado en el cuaderno y con todas las alertas del informe:

| Problema o alerta | Variable(s) | ¿Lo detecta el informe? | ¿Lo habías detectado tú? ¿En qué ejercicio? | ¿Es un problema real? |
|---|---|---|---|---|
| | | | | |
| | | | | |
| | | | | |
| | | | | |
| | | | | |

**Preguntas:**

1. ¿Detecta el informe que los ceros de `Cholesterol` son en realidad valores ausentes? ¿Qué dice exactamente sobre esa variable?
2. ¿Qué cosas descubriste tú a mano que el informe **no** señala? Piensa en los ejercicios 1.3 y 2.4.
3. ¿Hay alguna alerta del informe que **no** sea un problema real? ¿Cuál y por qué?
4. Con lo que has visto, ¿puede el perfilado automático sustituir al AED manual? ¿Para qué es más útil?

*Escribe aquí tus respuestas:*

## Ejercicio 3.4 — Del gráfico a la conclusión: ficha de síntesis

**Parte A — Tres preguntas, tres gráficos.**
Responde a cada pregunta con **un gráfico** y, debajo, una celda de texto con tu interpretación (2-4 frases que respondan a la pregunta con cifras).

**Pregunta 1.** ¿A partir de qué edad aumenta claramente el riesgo de enfermedad? Agrupa la edad en tramos de 10 años con `pd.cut()` y representa el porcentaje de enfermos en cada tramo, indicando cuántos pacientes hay en cada uno.

In [ ]:
# TODO: pregunta 1

*Interpretación:*

**Pregunta 2.** La angina de esfuerzo (`ExerciseAngina`) y la pendiente ST (`ST_Slope`) salen de la misma prueba. ¿Aportan información distinta o dicen lo mismo? Dibuja un mapa de calor con el **porcentaje de enfermos** en cada combinación de las dos variables, y escribe en cada celda también el número de pacientes.

In [ ]:
# TODO: pregunta 2

*Interpretación:*

**Pregunta 3.** ¿Tener la glucosa en ayunas alta (`FastingBS = 1`) aumenta el riesgo igual en hombres que en mujeres?

In [ ]:
# TODO: pregunta 3

*Interpretación:*

**Parte B — Ficha de síntesis del dataset.**
Completa la ficha con lo que has aprendido en todo el cuaderno. Es el resumen que le darías a otra persona antes de que empiece a limpiar los datos y a entrenar un modelo.

| Apartado | Contenido |
|---|---|
| **Dataset y origen** | |
| **Entidad (qué es cada fila)** | |
| **Volumetría (filas × columnas)** | |
| **Variable objetivo y equilibrio de clases** | |
| **Variables numéricas** | |
| **Variables categóricas (nominales / ordinales / binarias)** | |
| **Cobertura: ¿a quién representa y a quién no?** | |

**Problemas de calidad que hay que tratar en el Bloque 3:**

| Variable | Problema | Nº de filas afectadas | Qué propondrías hacer |
|---|---|---|---|
| | | | |
| | | | |
| | | | |

**Selección de atributos.** Las **5 variables** que crees más útiles para predecir `HeartDisease`, de más a menos, con la evidencia (ejercicio y cifra) que lo justifica:

| # | Variable | Evidencia |
|---|---|---|
| 1 | | |
| 2 | | |
| 3 | | |
| 4 | | |
| 5 | | |

¿Hay alguna variable que **no** usarías, o que usarías con precaución? ¿Por qué?

*Escribe aquí tus respuestas:*